In [39]:
import numpy as np
from collections import defaultdict
import pandas as pd

In [40]:
import os
print(os.getcwd())

c:\Users\koolr\OneDrive\ROHIT\Work and Resume\Career Learning\Fifa Project


In [41]:
elo_df = pd.read_csv("data/elo_ratings.csv")
elo= dict(zip(elo_df["teams"], elo_df["elo"]))

print(elo["Brazil"])
print(elo["France"])

1865.52
1890.57


In [42]:
import os
print(os.listdir("data"))

['elo_ratings.csv', 'features.csv', 'rf_model.pkl']


In [43]:
def match_probabilities(home, away):
    ra= elo.get(home, 1500)
    rb= elo.get(away, 1500)
    e= 1/ (1+10**((rb-ra)/400))
    draw=0.25
    win= e*(1-draw)
    loss= (1-e)*(1-draw)
    return win, draw, loss

In [44]:
print(match_probabilities("Brazil", "Honduras"))

(0.6179587853246128, 0.25, 0.13204121467538724)


In [45]:
def simulate_match(home, away):
    win, draw, loss= match_probabilities(home, away)
    outcome= np.random.choice(["W", "D", "L"], p=[win, draw, loss])
    return outcome

In [46]:
result = simulate_match("Brazil", "Honduras")
print(result)

W


In [47]:
groups = {
    "A": ["Mexico", "South Korea", "Czechia", "South Africa"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Scotland", "Haiti"],
    "D": ["United States", "Australia", "Turkey", "Paraguay"],
    "E": ["Germany", "Ivory Coast", "Ecuador", "Curacao"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Uruguay", "Saudi Arabia", "Cape Verde"],
    "I": ["France", "Senegal", "Norway", "Iraq"],
    "J": ["Argentina", "Austria", "Algeria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

In [48]:
print(elo_df.head(20))

          teams      elo  rank
0     Argentina  1955.62     1
1         Spain  1948.68     2
2        France  1890.57     3
3       Morocco  1878.77     4
4        Brazil  1865.52     5
5         Japan  1862.70     6
6      Portugal  1862.46     7
7       England  1853.22     8
8      Colombia  1847.46     9
9       Germany  1846.94    10
10  Netherlands  1823.84    11
11       Mexico  1816.06    12
12      Ecuador  1814.82    13
13      Algeria  1808.67    14
14         Iran  1803.05    15
15      Croatia  1792.02    16
16      Senegal  1791.27    17
17        Italy  1790.94    18
18      Belgium  1790.32    19
19  South Korea  1786.27    20


In [49]:
print(groups["A"])
print(groups["A"][0])

['Mexico', 'South Korea', 'Czechia', 'South Africa']
Mexico


In [50]:
def simulate_group(group_letter):
    teams=groups[group_letter]
    points = defaultdict(int)
    for i in range (len(teams)):
        for j in range(i+1, len(teams)):
            home = teams[i]
            away = teams[j]
            outcome = simulate_match(home, away)

            if outcome == "W":
                points[home]+=3
            elif outcome == "L":
                points[away]+=3
            else:
                points[home]+=1
                points[away]+=1

    standings= sorted(teams, key=lambda t: points[t], reverse=True)
    return standings

In [51]:
print(simulate_group("A"))

['Mexico', 'South Korea', 'South Africa', 'Czechia']


In [52]:
def simulate_group_stage():
    qualifiers= []
    third_place=[]

    for group_letter in groups: 
        standings= simulate_group(group_letter)
        qualifiers.append(standings[0])
        qualifiers.append(standings[1])
        third_place.append(standings[2])
    
    third_place= sorted(third_place, key=lambda t:elo.get(t,1500), reverse=True)[:8]
    qualifiers.extend(third_place)

    return qualifiers

In [53]:
result= simulate_group_stage()
print(len(result))
print(result)

32
['Mexico', 'South Korea', 'Canada', 'Switzerland', 'Brazil', 'Morocco', 'Australia', 'United States', 'Ivory Coast', 'Ecuador', 'Japan', 'Netherlands', 'Belgium', 'Iran', 'Uruguay', 'Spain', 'Senegal', 'Norway', 'Jordan', 'Argentina', 'Colombia', 'Portugal', 'England', 'Ghana', 'Germany', 'Turkey', 'Austria', 'Egypt', 'Uzbekistan', 'Panama', 'Iraq', 'Tunisia']


In [54]:
def simulate_knockout(qualifiers):
    teams= qualifiers[:]

    while len(teams) > 1:
        next_round= []
        for i in range(0, len(teams), 2):
            home= teams[i]
            away= teams[i+1]
            outcome= simulate_match(home,away)
            if outcome == "D":
                outcome= np.random.choice(["W", "L"])
            if outcome == "W":
                next_round.append(home)
            else: 
                next_round.append(away)
        teams= next_round
    
    return teams [0]

In [55]:
print(simulate_knockout(result))

South Korea


In [56]:
from collections import Counter

def run_monte_carlo(n=10000):
    win_counts = Counter()

    for i in range(n):
        qualifiers= simulate_group_stage()
        winner= simulate_knockout(qualifiers)
        win_counts[winner] += 1

    results= {team: count/n for team, count in win_counts.most_common()}

    return results

In [57]:
results = run_monte_carlo()
for team, prob in list(results.items())[:10]:
    print(f"{team}: {prob:.1%}")

Spain: 11.5%
Argentina: 11.2%
Morocco: 5.9%
France: 5.7%
Brazil: 4.8%
Portugal: 4.6%
Japan: 4.5%
England: 4.5%
Germany: 3.9%
Mexico: 3.5%
